# Del modelo a la API: `transformers`, `pipeline` e `InferenceClient`

**Seminario de Actualización · IFTS N.º 18 · Clase 7** — Docente: Cynthia M. Villagra

## Qué vas a poder hacer al terminar

1. Usar `pipeline` indicando **solo la tarea**, **solo el modelo** o **las dos cosas**, y explicar qué pasa en cada caso.
2. Abrir la caja: hacer a mano los **tres pasos** que `pipeline` hace por vos.
3. Hacer lo mismo con `InferenceClient` y señalar **dónde quedó cada paso**.
4. Decidir, para una app, **dónde conviene que corra el modelo**: en tu compu o en el servidor de un proveedor.

## Cómo está organizado

| Parte | Qué hacés | ¿Se bajan los pesos del modelo? |
|---|---|---|
| **A** | `transformers` y `pipeline` | **Sí**: el modelo corre en tu compu (o en Colab) |
| **B** | `InferenceClient` | **No**: el modelo corre en los servidores de Hugging Face |
| **C** | Comparación lado a lado | — |
| **D** | Actividad: cuatro apps (Gradio y Streamlit, con cada enfoque) | — |

Cada ejemplo sigue el mismo ritmo: **Predicción** (pensá qué va a pasar antes de correr) → **Ejecutá** → **Qué mirar**.

**Dónde correrlo**: en Google Colab (recomendado si tu compu del trabajo no te deja instalar) o en tu compu con un entorno virtual. La Parte D (Streamlit) conviene hacerla en tu compu o en Codespaces.

## El mapa de toda la notebook

```
PARTE A · pipeline: tu compu hace todo
   "I love this" ──► [tokenizador] ──► [modelo + pesos] ──► [softmax + etiquetas] ──► POSITIVE 99%
                      texto → números    números → números    números → respuesta
                      └──────────────── todo en TU compu (bajaste los pesos) ───────────────┘

PARTE B · InferenceClient: tu compu solo pregunta
   "I love this" ──── internet + tu token ────► servidor de Hugging Face
                                                [tokenizador] → [modelo + pesos] → [softmax + etiquetas]
   POSITIVE 99% ◄──────────── respuesta ────────────┘
```

Son **los mismos tres pasos** en los dos casos. Lo único que cambia es **en qué computadora pasan**.

## Glosario (español ↔ inglés)

La documentación está en inglés: estos son los términos que vas a cruzar.

| En español | En inglés | Qué es |
|---|---|---|
| modelo | *model* | el programa ya entrenado que hace la predicción |
| pesos | *weights* | los números que el modelo aprendió al entrenarse; son lo que se descarga |
| tarea | *task* | qué problema resuelve: clasificar texto, completar una palabra, etc. |
| tokenizador | *tokenizer* | parte el texto en pedazos y los convierte en números |
| token (del texto) | *token* | cada pedazo de texto convertido en número |
| logits | *logits* | los números "crudos" que devuelve el modelo, uno por etiqueta |
| etiqueta | *label* | la categoría de la respuesta: `POSITIVE`, `joy`, etc. |
| puntaje | *score* | la probabilidad (de 0 a 1) de esa etiqueta |
| inferencia | *inference* | usar un modelo ya entrenado para predecir. No lo modifica |
| caché | *cache* | la carpeta donde quedan guardados los modelos que bajaste |
| proveedor | *provider* | la empresa cuyos servidores corren el modelo por vos |
| token de acceso | *access token* | tu "contraseña" para la API. **Ojo: no tiene nada que ver con el token del texto** |

## Los modelos de esta notebook

Todos livianos (menos de 350 MB) y disponibles también en la Inference API de Hugging Face (verificado el 02/10/2026).

| Modelo | Tarea | Tamaño de los pesos | Idioma |
|---|---|---|---|
| `distilbert/distilbert-base-uncased-finetuned-sst-2-english` | clasificar sentimiento (2 etiquetas) | 268 MB | inglés |
| `j-hartmann/emotion-english-distilroberta-base` | clasificar emociones (7 etiquetas) | 329 MB | inglés |
| `distilbert/distilbert-base-uncased` | completar la palabra que falta (*fill-mask*) | 268 MB | inglés |
| `typeform/distilbert-base-uncased-mnli` | clasificar con **tus** etiquetas (*zero-shot*) | 268 MB | inglés |

Son modelos entrenados en inglés: por eso los ejemplos están en inglés. Probar después con frases en español es parte de la actividad.

---
# PARTE A · `transformers` y `pipeline`: el modelo corre en tu compu

## A0 · Instalación

`transformers` es la librería de Hugging Face para usar modelos. `torch` (PyTorch) es el motor que hace las cuentas. En Colab ya vienen instaladas; en tu compu, corré esta celda una vez.

In [ ]:
# En Colab no hace falta. En tu compu (con el entorno virtual activado):
%pip install -q transformers torch

## A1 · Solo la tarea

**Predicción**: le vamos a decir a `pipeline` *qué* queremos hacer ("análisis de sentimiento") pero **no** con qué modelo. ¿Qué creés que va a pasar? ¿Error? ¿Elige uno solo? ¿Cuál?

In [ ]:
from transformers import pipeline

pipe = pipeline("sentiment-analysis")      # solo la tarea
pipe("I love this")

**Qué mirar**

- Aparece una advertencia: *No model was supplied, defaulted to `distilbert/distilbert-base-uncased-finetuned-sst-2-english`*. Como no le dijiste qué modelo usar, eligió **el modelo por defecto** de esa tarea y **bajó sus pesos** (la primera vez tarda).
- `"sentiment-analysis"` es un alias: el nombre oficial de la tarea es `"text-classification"`.
- La respuesta es una **lista con un diccionario adentro**: `[{'label': ..., 'score': ...}]`.

**Por qué no conviene**: el modelo por defecto lo decide la librería y puede cambiar en otra versión. Tu app podría cambiar de comportamiento sin que toques nada. **Siempre indicá el modelo.**

In [ ]:
print(type(pipeline))   # pipeline es una FUNCIÓN...
print(type(pipe))       # ...que devuelve un OBJETO (lo vimos en Clase 6)
print(pipe.task)        # qué tarea quedó configurada

## A2 · Solo el modelo

**Predicción**: ahora al revés, le damos **el modelo** pero **no la tarea**. ¿Cómo podría saber `pipeline` qué hacer con ese modelo?

In [ ]:
pipe_emo = pipeline(model="j-hartmann/emotion-english-distilroberta-base")   # solo el modelo
print(pipe_emo.task)
pipe_emo("I love this")

**Qué mirar**

- Funcionó: `pipeline` le preguntó al Hub qué tarea declara ese modelo en su *model card* (el campo `pipeline_tag`) y se configuró solo como `text-classification`.
- Este modelo tiene **otras etiquetas**: emociones (`joy`, `anger`, `sadness`, `fear`, `surprise`, `disgust`, `neutral`). **Las etiquetas las define el modelo, no la tarea.**
- Es cómodo, pero depende de que el autor del modelo haya completado bien su model card.

## A3 · Tarea + modelo: la forma recomendada

Así es como lo vamos a escribir de acá en adelante: **qué hacer** y **con qué**, explícitos.

In [ ]:
pipe = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

pipe("I love this")

**Predicción**: devuelve una sola etiqueta, la ganadora. ¿Qué pasará con la otra? ¿Cuánto creés que vale su puntaje?

In [ ]:
pipe("I love this", top_k=None)      # top_k=None: todas las etiquetas, no solo la ganadora

**Qué mirar**: los puntajes de todas las etiquetas **suman 1**: son probabilidades. La "respuesta" es simplemente la etiqueta con el puntaje más alto.

In [ ]:
frases = ["I love this", "This is terrible", "Not bad at all", "Great, the server crashed again"]
for frase, r in zip(frases, pipe(frases)):          # también acepta una lista de textos
    print(f"{frase:35} → {r['label']} ({r['score']:.0%})")

**Ahora pensá**: ¿alguna te sorprendió? *"Great, the server crashed again"* es irónica. El modelo no "entiende": aprendió patrones de las frases con las que se entrenó.

## A4 · Misma tarea, otro modelo

**Predicción**: es la misma tarea (`text-classification`) y las mismas frases. Si cambio solo el modelo, ¿qué cambia en el resultado?

In [ ]:
pipe_emo = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")
for frase, r in zip(frases, pipe_emo(frases)):
    print(f"{frase:35} → {r['label']} ({r['score']:.0%})")

**Qué mirar**: cambiar de modelo es **cambiar un texto**. El código es idéntico; lo que cambia son las etiquetas posibles y cómo clasifica. Esta idea va a ser clave en la actividad.

## A5 · Otra tarea: completar la palabra que falta (*fill-mask*)

Este modelo aprendió a adivinar una palabra tapada con `[MASK]`. **Predicción**: ¿qué palabra va a proponer en primer lugar?

In [ ]:
pipe_mask = pipeline("fill-mask", model="distilbert/distilbert-base-uncased")

for r in pipe_mask("Paris is the [MASK] of France."):
    print(f"{r['token_str']:12} {r['score']:.0%}   → {r['sequence']}")

**Qué mirar**

- Otra tarea, **otra forma de respuesta**: ahora cada diccionario trae `token_str` (la palabra), `score` y `sequence` (la frase completa).
- Por defecto devuelve las **5** más probables.
- La marca `[MASK]` es la de este modelo. Otros modelos usan otra (`<mask>`, por ejemplo): se puede consultar con `pipe_mask.tokenizer.mask_token`.

## A6 · Otra tarea: clasificar con tus propias etiquetas (*zero-shot*)

Hasta ahora las etiquetas venían fijas en el modelo. Con *zero-shot* **las elegís vos** al momento de usarlo.

**Predicción**: ¿en qué categoría va a poner la frase?

In [ ]:
pipe_zs = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli")

r = pipe_zs(
    "My laptop does not turn on since yesterday",
    candidate_labels=["technical problem", "billing", "praise"],
)
r

**Qué mirar**: la forma de la respuesta **es distinta otra vez**: un solo diccionario con dos **listas paralelas**, `labels` y `scores`, ordenadas de mayor a menor. Cada tarea tiene su formato de salida: siempre conviene imprimir la respuesta antes de usarla.

In [ ]:
for etiqueta, puntaje in zip(r["labels"], r["scores"]):
    print(f"{etiqueta:20} {puntaje:.0%}")

## A7 · Abrir la caja: los tres pasos de `pipeline`, a mano

`pipeline` hace tres pasos por vos. Ahora los hacemos uno por uno, con las piezas de `transformers`. **No hace falta memorizar este código**: la idea es ver qué hay adentro.

### Paso 1 · Texto → números (tokenizador)

**Predicción**: cuando le paso "I love this" al modelo, ¿qué le llega? ¿El texto?

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODELO = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tokenizador = AutoTokenizer.from_pretrained(MODELO)

entrada = tokenizador("I love this", return_tensors="pt")      # pt = formato de PyTorch
print(entrada["input_ids"])
print(tokenizador.convert_ids_to_tokens(entrada["input_ids"][0]))

**Qué mirar**: el modelo nunca ve letras. Recibe números: uno por cada pedazo del texto, más dos marcas especiales, `[CLS]` (inicio) y `[SEP]` (fin). Todo quedó en minúscula porque el modelo es *uncased*.

### Paso 2 · Números → números (el modelo con sus pesos)

In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO)

with torch.no_grad():                 # solo inferencia: no vamos a entrenar
    logits = modelo(**entrada).logits
logits

**Qué mirar**: el modelo devuelve **dos números**, uno por etiqueta. Son los *logits*: todavía no son probabilidades (pueden ser negativos y no suman 1).

### Paso 3 · Números → respuesta (softmax + etiquetas)

In [ ]:
probabilidades = torch.softmax(logits, dim=-1)[0]      # softmax: convierte en probabilidades que suman 1
print(probabilidades)
print(modelo.config.id2label)                            # qué etiqueta corresponde a cada posición

ganadora = int(probabilidades.argmax())
print(modelo.config.id2label[ganadora], f"{float(probabilidades[ganadora]):.0%}")

**Qué mirar**: el resultado es **el mismo** que dio `pipe("I love this")` en A3. Esas tres celdas son lo que `pipeline` hace en una línea.

| Paso | A mano | Dentro de `pipeline` |
|---|---|---|
| 1. Texto → números | `tokenizador(texto)` | `pipe.tokenizer` |
| 2. Números → números | `modelo(**entrada).logits` | `pipe.model` |
| 3. Números → respuesta | `softmax` + `id2label` | el post-procesamiento de `pipe` |

In [ ]:
# Comprobalo: el pipeline tiene adentro el mismo tipo de piezas
print(type(pipe.tokenizer).__name__)
print(type(pipe.model).__name__)

## A8 · ¿Dónde quedaron los pesos?

Cada modelo que usaste **se descargó a tu disco** y quedó en la caché. La segunda vez que lo pedís, no se vuelve a bajar.

In [ ]:
from huggingface_hub import scan_cache_dir

cache = scan_cache_dir()
for repo in cache.repos:
    print(f"{repo.repo_id:55} {repo.size_on_disk / 1e6:7.0f} MB")
print(f"\nTotal en disco: {cache.size_on_disk / 1e6:.0f} MB")

**Ahora pensá**: sumá lo que ocupan los modelos **y** `torch`. Si quisieras publicar esta app en un servidor gratis con 512 MB de memoria (como Render), ¿entraría? Lo retomamos en la Parte B.

## A9 · Empaquetarlo en una función propia

Lo que vamos a reusar en las apps: una función nuestra, con el pipeline creado **afuera** (se carga una sola vez).

In [ ]:
from transformers import pipeline

pipe = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")   # afuera: se carga UNA vez

def clasificar(texto):
    out = pipe(texto)
    return f"{out[0]['label']} ({out[0]['score']:.0%})"

clasificar("I love this")

---
# PARTE B · `InferenceClient`: el modelo corre en los servidores de Hugging Face

Mismos modelos, mismas tareas, mismas frases. La diferencia: **no se baja nada**. Tu código manda el texto por internet y recibe la respuesta.

## B0 · Instalación y token de acceso

- `huggingface_hub` es liviana: **no necesita `torch`**.
- Necesitás un **token de acceso** de Hugging Face: *Settings → Access Tokens → New token*, con el permiso **"Make calls to Inference Providers"**.
- **El token nunca va escrito en el código.** Esta celda te lo pide sin mostrarlo, o lo toma de la variable de entorno `HF_TOKEN` si ya la tenés.

In [ ]:
%pip install -q huggingface_hub

In [ ]:
import os
from getpass import getpass

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Pegá tu token de Hugging Face (no se va a ver): ")

## B1 · El cliente: una clase que instanciás

**Predicción**: con lo que vimos en Clase 6: `InferenceClient`, ¿es una función o una clase? ¿Y `client`?

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(provider="hf-inference", api_key=os.environ["HF_TOKEN"])
print(type(client))

**Qué mirar**: `InferenceClient` es una **clase** (PascalCase); `client` es el **objeto**. No se descargó ningún modelo: el objeto solo sabe **a quién preguntarle** (`provider`) y **con qué credencial** (`api_key`).

## B2 · Tarea + modelo (el equivalente de A3)

**Predicción**: ¿la respuesta va a ser igual a la de A3? ¿Igual en el contenido? ¿Y en la forma?

In [ ]:
resultado = client.text_classification("I love this", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
resultado

**Qué mirar**

- **La tarea ahora es el nombre del método**: `text_classification`. En `pipeline` era un texto, `"text-classification"`.
- **El modelo se pasa en cada llamado** con `model=`.
- La respuesta es una **lista de objetos**, no de diccionarios: se leen con punto, `r.label` y `r.score`.
- ¿Y el tokenizador, el modelo y el softmax de A7? **Pasaron en el servidor.** Están, pero no en tu compu.

In [ ]:
for r in resultado:
    print(r.label, f"{r.score:.0%}")

### ¿Dónde quedaron las piezas de A7?

| Paso | Con `pipeline` (Parte A) | Con `InferenceClient` (Parte B) |
|---|---|---|
| 1. Texto → números | `pipe.tokenizer`, en tu compu | en el servidor de Hugging Face |
| 2. Números → números | `pipe.model` + pesos descargados, en tu compu | en el servidor (los pesos están allá) |
| 3. Números → respuesta | post-procesamiento, en tu compu | en el servidor |
| Lo que hace tu compu | **todo** | mandar el texto y recibir la respuesta |

## B3 · Solo el modelo (el equivalente de A2)

En `pipeline` podías dar solo el modelo. Con `InferenceClient`, el equivalente es **fijar el modelo al crear el cliente**: queda guardado en el objeto y no hace falta repetirlo.

In [ ]:
client_emo = InferenceClient(model="j-hartmann/emotion-english-distilroberta-base", provider="hf-inference", api_key=os.environ["HF_TOKEN"])

for frase in frases:
    r = client_emo.text_classification(frase)
    print(f"{frase:35} → {r[0].label} ({r[0].score:.0%})")

**Qué mirar**: lo que no cambia es **la tarea**: la elegís vos con el método. No existe un "método genérico" que adivine la tarea como hacía `pipeline(model=...)`.

**¿Y "solo la tarea", como en A1?** También se puede llamar sin modelo, y la librería elige uno recomendado. **Mismo consejo que en A1: indicalo siempre.**

## B4 · Todas las etiquetas (el equivalente de `top_k=None`)

In [ ]:
client.text_classification("I love this", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", top_k=2)

## B5 · *Fill-mask* (el equivalente de A5)

**Predicción**: ¿la palabra que propone va a ser la misma que en A5?

In [ ]:
for r in client.fill_mask("Paris is the [MASK] of France.", model="distilbert/distilbert-base-uncased"):
    print(f"{r.token_str:12} {r.score:.0%}   → {r.sequence}")

## B6 · *Zero-shot* (el equivalente de A6)

In [ ]:
resultado = client.zero_shot_classification(
    "My laptop does not turn on since yesterday",
    candidate_labels=["technical problem", "billing", "praise"],
    model="typeform/distilbert-base-uncased-mnli",
)
for r in resultado:
    print(f"{r.label:20} {r.score:.0%}")

**Qué mirar**: con `pipeline` la respuesta de *zero-shot* eran **dos listas paralelas** (`labels`, `scores`). Con `InferenceClient` es una **lista de objetos**, cada uno con `label` y `score`. Mismo contenido, otra forma: **si copiás código de un enfoque al otro, se rompe**.

## B7 · Dos errores que conviene reconocer

**Predicción**: los dos errores de abajo parecen lo mismo ("algo del token"). ¿Quién da cada uno, tu compu o Hugging Face?

In [ ]:
# Error 1: la variable no existe
try:
    os.environ["TOKEN_QUE_NO_EXISTE"]
except KeyError as e:
    print("KeyError:", e)

In [ ]:
# Error 2: el token está mal
cliente_malo = InferenceClient(provider="hf-inference", api_key="hf_token_inventado")
try:
    cliente_malo.text_classification("I love this", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
except Exception as e:
    print(type(e).__name__, "→", str(e)[:120])

**Qué mirar**

- El `KeyError` lo da **tu compu**: el pedido ni siquiera salió.
- El **401 Unauthorized** lo da **Hugging Face**: el pedido llegó, pero la credencial no sirve.

## B8 · La misma función propia, con otro motor

Compará con A9: **la función se llama igual, recibe lo mismo y devuelve lo mismo**. Cambió solo lo de adentro. Por eso una app que usa `clasificar(texto)` no se entera del cambio.

In [ ]:
MODELO = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
client = InferenceClient(provider="hf-inference", api_key=os.environ["HF_TOKEN"])   # afuera, como el pipe

def clasificar(texto):
    out = client.text_classification(texto, model=MODELO)
    return f"{out[0].label} ({out[0].score:.0%})"

clasificar("I love this")

---
# PARTE C · Lado a lado

## C1 · Tareas de `pipeline` ↔ métodos de `InferenceClient`

| Tarea | Con `pipeline` | Con `InferenceClient` | Cómo se lee la respuesta |
|---|---|---|---|
| Clasificar texto | `pipeline("text-classification", model=M)(texto)` | `client.text_classification(texto, model=M)` | `r["label"]` ↔ `r.label` |
| Completar palabra | `pipeline("fill-mask", model=M)(texto)` | `client.fill_mask(texto, model=M)` | `r["token_str"]` ↔ `r.token_str` |
| Zero-shot | `pipeline("zero-shot-classification", model=M)(texto, candidate_labels=[...])` | `client.zero_shot_classification(texto, candidate_labels=[...], model=M)` | `r["labels"][i]` ↔ `r[i].label` |
| Reconocer entidades | `pipeline("token-classification", model=M, aggregation_strategy="simple")(texto)` | `client.token_classification(texto, model=M, aggregation_strategy="simple")` | `r["entity_group"]` ↔ `r.entity_group` |

**La regla**: la tarea con guiones (`"text-classification"`) pasa a ser un método con guion bajo (`text_classification`).

## C2 · Las diferencias que importan

| | `pipeline` (Parte A) | `InferenceClient` (Parte B) |
|---|---|---|
| Qué es | una **función** que devuelve un objeto | una **clase** que instanciás |
| Dónde corre el modelo | en tu compu (o en Colab) | en los servidores del proveedor |
| Qué instalás | `transformers` + `torch` (pesado) | `huggingface_hub` (liviano) |
| Se bajan los pesos | sí, a la caché | no |
| Qué necesitás | memoria y disco | internet y un token |
| La tarea se elige | con un texto: `"text-classification"` | con el método: `.text_classification()` |
| El modelo se elige | al crear el pipeline | en cada llamado, o al crear el cliente |
| Qué devuelve | diccionarios: `r["label"]` | objetos: `r.label` |
| La primera vez | lenta (descarga) | rápida, pero cada llamado depende de la red |
| Tus datos | no salen de tu compu | viajan al proveedor |
| Quién pone el límite | tu hardware | el proveedor: cuota y disponibilidad |

## C3 · ¿Cuál uso?

Ninguna es "mejor". Depende de **dónde corre tu app**:

- **Prototipo en tu compu o en Colab, o datos que no pueden salir** → `pipeline`.
- **App publicada en un servidor gratis con poca memoria (Render: 512 MB)** → `InferenceClient`: `torch` y los pesos no entran.
- **Servidor con memoria suficiente (pago)** → cualquiera de las dos: es una decisión de costo.

---
# PARTE D · Actividad: cuatro apps, un mismo núcleo

## La consigna

Armá **cuatro apps** que usen el mismo núcleo, una función `clasificar(texto)`:

| App | Motor | Interfaz | Archivo |
|---|---|---|---|
| 1 | `pipeline` | Gradio | `app_gradio_pipeline.py` |
| 2 | `pipeline` | Streamlit | `app_streamlit_pipeline.py` |
| 3 | `InferenceClient` | Gradio | `app_gradio_api.py` |
| 4 | `InferenceClient` | Streamlit | `app_streamlit_api.py` |

**Elegí vos** la tarea y el modelo, entre los de esta notebook (o cualquier otro liviano del Hub que esté disponible en la Inference API). Las cuatro apps tienen que usar **el mismo modelo**, así podés comparar.

## Qué se entrega

En la carpeta `Clase_07/` de tu portfolio:

1. Los cuatro archivos `.py`.
2. Dos `requirements.txt`: uno para las apps con `pipeline` y otro para las de `InferenceClient` (por ejemplo, en subcarpetas `local/` y `api/`).
3. Un `.gitignore` con `.env` adentro.
4. Un `README.md` que responda:
   - qué modelo y qué tarea elegiste, y por qué;
   - **dónde corre el modelo** en cada una de las cuatro apps;
   - qué tuviste que cambiar entre la app 1 y la app 3 (y qué **no** cambiaste);
   - opcional: una captura de cada app funcionando.
5. **Entrega**: el link al repositorio.

**Si no llegás con todo**: primero las dos de Gradio (1 y 3), después las de Streamlit.

**Extra (opcional)**: publicá la app 3 en Render (con `HF_TOKEN` en *Environment*) y poné el link en el README.

## Cómo correr cada una

| | Gradio | Streamlit |
|---|---|---|
| Comando | `python app_gradio_pipeline.py` | `streamlit run app_streamlit_pipeline.py` |
| Se abre en | `http://127.0.0.1:7860` | `http://localhost:8501` |
| Qué se ejecuta en cada interacción | solo la función | **todo el script**: hace falta `@st.cache_resource` |

El token, para las apps 3 y 4: en la terminal antes de correrlas (`$env:HF_TOKEN = "hf_..."` en PowerShell, `export HF_TOKEN="hf_..."` en Git Bash), o en un `.env` leído con `python-dotenv`.

## Esqueletos para arrancar

Completá los `# TODO`. La celda `%%writefile` guarda el contenido como archivo: después lo corrés desde la terminal.

In [ ]:
%%writefile app_gradio_pipeline.py
# App 1 · pipeline + Gradio: el modelo corre en TU compu
import gradio as gr
from transformers import pipeline

MODELO = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
pipe = pipeline("text-classification", model=MODELO)   # afuera: se carga una vez

def clasificar(texto):
    # TODO: llamar a pipe con el texto y devolver "ETIQUETA (porcentaje)"
    ...

demo = gr.Interface(fn=clasificar, inputs="text", outputs="text",
                    title="Clasificador (pipeline local)")
demo.launch()

In [ ]:
%%writefile app_streamlit_pipeline.py
# App 2 · pipeline + Streamlit: el modelo corre en TU compu
import streamlit as st
from transformers import pipeline

MODELO = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

# TODO: agregar el decorador que evita recargar el modelo en cada interacción
def cargar_pipe():
    return pipeline("text-classification", model=MODELO)

pipe = cargar_pipe()

st.title("Clasificador (pipeline local)")
texto = st.text_input("Escribí una frase")
if texto:
    # TODO: clasificar el texto y mostrar el resultado con st.write
    ...

In [ ]:
%%writefile app_gradio_api.py
# App 3 · InferenceClient + Gradio: el modelo corre en el servidor de Hugging Face
import os
import gradio as gr
from huggingface_hub import InferenceClient

MODELO = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
# TODO: crear el client con provider="hf-inference" y el token desde la variable de entorno

def clasificar(texto):
    # TODO: llamar al método de la tarea y devolver "ETIQUETA (porcentaje)"
    #       Ojo: la respuesta son objetos, se leen con punto.
    ...

demo = gr.Interface(fn=clasificar, inputs="text", outputs="text",
                    title="Clasificador (Inference API)")
demo.launch(server_name="0.0.0.0", server_port=int(os.environ.get("PORT", 7860)))   # listo para Render

In [ ]:
%%writefile app_streamlit_api.py
# App 4 · InferenceClient + Streamlit: el modelo corre en el servidor de Hugging Face
import os
import streamlit as st
from huggingface_hub import InferenceClient

MODELO = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

@st.cache_resource
def crear_cliente():
    # TODO: devolver un InferenceClient con provider="hf-inference" y el token
    ...

client = crear_cliente()

st.title("Clasificador (Inference API)")
texto = st.text_input("Escribí una frase")
if texto:
    # TODO: clasificar con client y mostrar el resultado
    ...

In [ ]:
%%writefile requirements_local.txt
transformers
torch
gradio
streamlit

In [ ]:
%%writefile requirements_api.txt
huggingface_hub
gradio
streamlit

## Antes de entregar, revisá

- [ ] Las cuatro apps usan el **mismo modelo** y la **misma función** `clasificar(texto)`.
- [ ] En las apps 1 y 2 el pipeline se crea **una sola vez** (afuera de la función, o con `@st.cache_resource`).
- [ ] En las apps 3 y 4 el token sale de una **variable de entorno**, nunca escrito en el código.
- [ ] `.env` está en el `.gitignore` y **no** aparece en `git status`.
- [ ] El `requirements` de las apps con API **no** tiene `torch` ni `transformers`.
- [ ] El README responde **dónde corre el modelo** en cada app.

## Para pensar (no se entrega)

1. Probá tus apps con una frase en español. ¿Qué pasa? ¿Qué te dice eso del modelo?
2. Si tu app 3 se hace pública y la usa mucha gente, ¿quién paga cada llamado?
3. Si tuvieras que clasificar datos de pacientes de un hospital, ¿cuál de los dos enfoques elegirías? ¿Por qué?

## Documentación oficial

- `pipeline`: https://huggingface.co/docs/transformers/main_classes/pipelines
- Tareas y modelos por defecto: https://huggingface.co/docs/transformers/task_summary
- `InferenceClient`: https://huggingface.co/docs/huggingface_hub/package_reference/inference_client
- Inference Providers: https://huggingface.co/docs/inference-providers
- Caché de modelos: https://huggingface.co/docs/huggingface_hub/guides/manage-cache
- Gradio `Interface`: https://www.gradio.app/docs/gradio/interface
- Streamlit `st.cache_resource`: https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource